In [4]:
import os
import sys
import random
from pathlib import Path
from typing import Tuple, List, Dict

from PIL import Image, ImageDraw, ImageFont
import pandas as pd
import numpy as np
import cv2

# ----------------------------
# Config (tweak as needed)
# ----------------------------
TARGET_SAMPLES = 150_000  # recommended: 100k-200k for CTC
TARGET_W, TARGET_H = 128, 32
MARGIN_HORIZONTAL = 80  # total left+right margin at 1x scale
MARGIN_VERTICAL = 40    # total top+bottom margin at 1x scale

UPPER_SYNTH_REL = Path("data") / "plate_upper_synth"
UPPER_SYNTH_DATA_SUBDIR = "data"
LABEL_COLUMNS = [
    "filename",
    "label",
    "source",
    "transactionDate",
    "plate",
    "province_code",
    "province_description",
    "brand_description",
    "colors_code",
    "colors_description",
    "vehicleClass",
]

RANDOM_SEED = 1337  # set None for non-deterministic runs
if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

# ----------------------------
# Project root detection
# ----------------------------
def find_project_root(start: Path) -> Path:
    """Find repo root so notebook works from synthetic_data/ or repo root."""
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "train" / "province_mapping.py").exists() and (p / "data").exists():
            return p
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / UPPER_SYNTH_REL
IMAGES_DIR = OUTPUT_DIR / UPPER_SYNTH_DATA_SUBDIR
LABELS_PATH = OUTPUT_DIR / "labels.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("IMAGES_DIR:", IMAGES_DIR)
print("LABELS_PATH:", LABELS_PATH)

# ----------------------------
# Fonts
# ----------------------------
def pick_font() -> ImageFont.ImageFont:
    preferred = "C:/Windows/Fonts/Sarun's ThangLuang.ttf"
    candidates = [
        preferred,
        "C:/Windows/Fonts/tahoma.ttf",
        "C:/Windows/Fonts/THSarabunNew.ttf",
        "C:/Windows/Fonts/THSarabunNew Bold.ttf",
    ]
    for path in candidates:
        if os.path.exists(path):
            try:
                return ImageFont.truetype(path, 38)  # slightly larger than lower line
            except OSError:
                continue
    print("Warning: Thai font not found, using default")
    return ImageFont.load_default()

font = pick_font()

# ----------------------------
# Palettes (reuse lower palette style)
# ----------------------------
palette = {
    "group1_personal": {
        "colors": [
            (255, 255, 255), (245, 245, 245), (230, 230, 230),
            (220, 225, 230), (210, 210, 210),
        ],
        "weight": 0.8,
    },
    "group2_taxi": {
        "colors": [
            (255, 235, 100), (255, 200, 90), (255, 180, 90),
            (150, 255, 150), (190, 240, 120),
        ],
        "weight": 0.1,
    },
    "group3_graphic": {
        "colors": [
            (255, 210, 230), (200, 220, 255), (255, 220, 170),
            (210, 200, 255), (180, 200, 210),
        ],
        "weight": 0.1,
    },
}

def choose_palette() -> Tuple[str, Tuple[int, int, int]]:
    groups = list(palette.keys())
    weights = [palette[g]["weight"] for g in groups]
    group = random.choices(groups, weights=weights, k=1)[0]
    color = random.choice(palette[group]["colors"])
    return group, color

# ----------------------------
# Character sets (uniform sampling per position)
# ----------------------------
DIGITS = list("0123456789")
THAI_CONSONANTS = [
    "ก", "ข", "ค", "ฆ", "ง", "จ", "ฉ", "ช", "ซ", "ฌ", "ญ",
    "ฎ", "ฏ", "ฐ", "ฑ", "ฒ", "ณ", "ด", "ต", "ถ", "ท",
    "ธ", "น", "บ", "ป", "ผ", "ฝ", "พ", "ฟ", "ภ", "ม",
    "ย", "ร", "ล", "ว", "ศ", "ษ", "ส", "ห", "ฬ", "อ", "ฮ",
]  # no ฅ ฃ

PREFIX_DIGIT_PROB = 0.2
LETTER_LEN_WEIGHTS = [(2, 0.8), (1, 0.1), (3, 0.1)]
SUFFIX_DIGIT_WEIGHTS = [(4, 0.75), (3, 0.2), (2, 0.05)]

def sample_plate_text() -> str:
    prefix = "" if random.random() > PREFIX_DIGIT_PROB else random.choice(DIGITS)
    letter_len = random.choices([l for l, _ in LETTER_LEN_WEIGHTS], weights=[w for _, w in LETTER_LEN_WEIGHTS], k=1)[0]
    letters = "".join(random.choice(THAI_CONSONANTS) for _ in range(letter_len))
    suffix_len = random.choices([l for l, _ in SUFFIX_DIGIT_WEIGHTS], weights=[w for _, w in SUFFIX_DIGIT_WEIGHTS], k=1)[0]
    digits = "".join(random.choice(DIGITS) for _ in range(suffix_len))
    return f"{prefix}{letters}{digits}"

# ----------------------------
# Drawing + augmentations
# ----------------------------
def draw_upper_line(text: str, bg_color: Tuple[int, int, int]) -> Image.Image:
    temp_img = Image.new("RGB", (1, 1))
    temp_draw = ImageDraw.Draw(temp_img)
    bbox = temp_draw.textbbox((0, 0), text, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]

    w = text_w + MARGIN_HORIZONTAL + random.randint(-5, 10)
    h = text_h + MARGIN_VERTICAL + random.randint(-3, 6)

    img = Image.new("RGB", (w, h), bg_color)
    draw = ImageDraw.Draw(img)
    draw.text((w // 2, h // 2), text, fill=(40, 40, 40), font=font, anchor="mm")
    return img

def apply_homography(img: np.ndarray) -> np.ndarray:
    height, width = img.shape[:2]
    src_points = np.float32([[0, 0], [width, 0], [width, height], [0, height]])
    distortion = random.uniform(0.03, 0.08)
    dst_points = np.float32([
        [random.uniform(0, width * distortion), random.uniform(0, height * distortion)],
        [width - random.uniform(0, width * distortion), random.uniform(0, height * distortion)],
        [width - random.uniform(0, width * distortion), height - random.uniform(0, height * distortion)],
        [random.uniform(0, width * distortion), height - random.uniform(0, height * distortion)],
    ])
    matrix = cv2.getPerspectiveTransform(src_points, dst_points)
    return cv2.warpPerspective(img, matrix, (width, height), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

def apply_augmentation(img: np.ndarray) -> np.ndarray:
    height, width = img.shape[:2]
    angle = random.uniform(-10, 10)
    rotation_matrix = cv2.getRotationMatrix2D((width / 2, height / 2), angle, 1.0)
    img = cv2.warpAffine(img, rotation_matrix, (width, height), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

    scale = random.uniform(0.85, 1.15)
    new_width = max(1, int(width * scale))
    new_height = max(1, int(height * scale))
    img = cv2.resize(img, (new_width, new_height), interpolation=cv2.INTER_CUBIC)

    if scale > 1.0:
        start_x = (new_width - width) // 2
        start_y = (new_height - height) // 2
        img = img[start_y : start_y + height, start_x : start_x + width]
    else:
        pad_x = (width - new_width) // 2
        pad_y = (height - new_height) // 2
        img = cv2.copyMakeBorder(
            img,
            pad_y,
            height - new_height - pad_y,
            pad_x,
            width - new_width - pad_x,
            cv2.BORDER_REPLICATE,
        )

    tx = random.randint(-18, 18)
    ty = random.randint(-8, 8)
    translation_matrix = np.float32([[1, 0, tx], [0, 1, ty]])
    img = cv2.warpAffine(img, translation_matrix, (width, height), borderMode=cv2.BORDER_REPLICATE)
    return img

def add_noise_and_blur(img: np.ndarray) -> np.ndarray:
    if random.random() < 0.3:
        if img.ndim == 3:
            noise = np.random.normal(0, 10, (img.shape[0], img.shape[1], 1)).astype(np.int16)
        else:
            noise = np.random.normal(0, 10, img.shape).astype(np.int16)
        img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    if random.random() < 1.0:
        kernel_size = random.choice([3, 5])
        img = cv2.GaussianBlur(img, (kernel_size, kernel_size), 0)

    if random.random() < 0.3:
        brightness = random.uniform(0.8, 1.2)
        img = cv2.convertScaleAbs(img, alpha=brightness, beta=0)
    return img

def apply_bicubic_interpolation(img: np.ndarray, scale: float = 0.5) -> np.ndarray:
    height, width = img.shape[:2]
    small = cv2.resize(img, (max(1, int(width * scale)), max(1, int(height * scale))), interpolation=cv2.INTER_CUBIC)
    return cv2.resize(small, (width, height), interpolation=cv2.INTER_CUBIC)

def thicken_text(image: np.ndarray) -> np.ndarray:
    k_size = 2
    kernel = np.ones((k_size, k_size), np.uint8)
    return cv2.erode(image, kernel, iterations=1)

def apply_heavy_motion_blur(image: np.ndarray) -> np.ndarray:
    kernel_size = random.choice([5, 7, 9, 11])
    kernel = np.zeros((kernel_size, kernel_size))
    kernel[int((kernel_size - 1) / 2), :] = np.ones(kernel_size)
    kernel /= kernel_size
    return cv2.filter2D(image, -1, kernel)

def fade_image(image: np.ndarray) -> np.ndarray:
    h, w, c = image.shape
    gray_val = random.randint(100, 180)
    gray_overlay = np.full((h, w, c), gray_val, dtype=np.uint8)
    alpha = random.uniform(0.1, 0.25)
    return cv2.addWeighted(image, 1 - alpha, gray_overlay, alpha, 0)

def apply_squash(img: np.ndarray) -> np.ndarray:
    h, w = img.shape[:2]
    squash_factor = random.uniform(0.6, 0.82)
    new_h = max(1, int(h * squash_factor))
    return cv2.resize(img, (w, new_h), interpolation=cv2.INTER_AREA)

# ----------------------------
# CSV helpers
# ----------------------------
def write_labels_header_if_missing():
    if not LABELS_PATH.exists():
        pd.DataFrame([], columns=LABEL_COLUMNS).to_csv(LABELS_PATH, index=False, encoding="utf-8-sig")

def append_rows(rows: List[Dict[str, object]]):
    if not rows:
        return
    df_new = pd.DataFrame(rows)
    for col in LABEL_COLUMNS:
        if col not in df_new.columns:
            df_new[col] = ""
    df_new = df_new[LABEL_COLUMNS]
    df_new.to_csv(LABELS_PATH, mode="a", header=False, index=False, encoding="utf-8-sig")

def existing_synth_count() -> int:
    return len(list(IMAGES_DIR.glob("synth_upper_*.jpg")))

# ----------------------------
# Main generation loop
# ----------------------------
def generate_upper_synthetic(n_samples: int = TARGET_SAMPLES, batch_flush: int = 2000):
    write_labels_header_if_missing()
    start_idx = existing_synth_count()
    current_idx = start_idx
    buffer: List[Dict[str, object]] = []
    generated = 0

    print(f"Starting from index: {current_idx}")
    print(f"Planned samples: {n_samples:,}")

    for _ in range(n_samples):
        while (IMAGES_DIR / f"synth_upper_{current_idx:06d}.jpg").exists():
            current_idx += 1

        plate_text = sample_plate_text()
        _, bg = choose_palette()
        img_pil = draw_upper_line(plate_text, bg)
        img_np = np.array(img_pil)

        img_np = apply_squash(img_np)
        if random.random() < 0.55:
            img_np = apply_homography(img_np)
        if random.random() < 0.65:
            img_np = apply_augmentation(img_np)
        if random.random() < 0.35:
            img_np = thicken_text(img_np)
        if random.random() < 0.8:
            img_np = apply_heavy_motion_blur(img_np)
        elif random.random() < 0.3:
            img_np = add_noise_and_blur(img_np)
        if random.random() < 0.65:
            img_np = fade_image(img_np)
        if random.random() < 0.3:
            img_np = add_noise_and_blur(img_np)
        if random.random() < 0.35:
            img_np = apply_bicubic_interpolation(img_np, scale=random.uniform(0.5, 0.85))

        img_np = cv2.resize(img_np, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

        fname = f"synth_upper_{current_idx:06d}.jpg"
        out_path = IMAGES_DIR / fname
        cv2.imwrite(str(out_path), cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR))

        buffer.append({
            "filename": fname,
            "label": plate_text,
            "source": "synthetic_upper",
            "transactionDate": "",
            "plate": plate_text,
            "province_code": "",
            "province_description": "",
            "brand_description": "",
            "colors_code": "",
            "colors_description": "",
            "vehicleClass": "1.0",
        })

        generated += 1
        current_idx += 1

        if len(buffer) >= batch_flush:
            append_rows(buffer)
            print(f"Flushed {len(buffer)} rows (total written: {generated:,})")
            buffer = []

    append_rows(buffer)
    print(f"Done. Generated total: {generated:,}")
    print(f"Images written to: {IMAGES_DIR}")
    print(f"Labels appended to: {LABELS_PATH}")

# Uncomment to run
generate_upper_synthetic(TARGET_SAMPLES)

PROJECT_ROOT: D:\CodingD\ALPR
OUTPUT_DIR: D:\CodingD\ALPR\data\plate_upper_synth
IMAGES_DIR: D:\CodingD\ALPR\data\plate_upper_synth\data
LABELS_PATH: D:\CodingD\ALPR\data\plate_upper_synth\labels.csv
Starting from index: 0
Planned samples: 150,000
Flushed 2000 rows (total written: 2,000)
Flushed 2000 rows (total written: 4,000)
Flushed 2000 rows (total written: 6,000)
Flushed 2000 rows (total written: 8,000)
Flushed 2000 rows (total written: 10,000)
Flushed 2000 rows (total written: 12,000)
Flushed 2000 rows (total written: 14,000)
Flushed 2000 rows (total written: 16,000)
Flushed 2000 rows (total written: 18,000)
Flushed 2000 rows (total written: 20,000)
Flushed 2000 rows (total written: 22,000)
Flushed 2000 rows (total written: 24,000)
Flushed 2000 rows (total written: 26,000)
Flushed 2000 rows (total written: 28,000)
Flushed 2000 rows (total written: 30,000)
Flushed 2000 rows (total written: 32,000)
Flushed 2000 rows (total written: 34,000)
Flushed 2000 rows (total written: 36,000)


In [1]:
# Test - Data

import os
import sys
import random
from pathlib import Path
from typing import Tuple, List, Dict

from PIL import Image, ImageDraw, ImageFont
import pandas as pd
import numpy as np
import cv2

# ----------------------------
# Config (tweak as needed)
# ----------------------------
TARGET_SAMPLES = 5000  # recommended: 100k-200k for CTC
TARGET_W, TARGET_H = 128, 32
MARGIN_HORIZONTAL = 80  # total left+right margin at 1x scale
MARGIN_VERTICAL = 40    # total top+bottom margin at 1x scale

UPPER_SYNTH_REL = Path("data") / "plate_upper_synth_test"
UPPER_SYNTH_DATA_SUBDIR = "data"
LABEL_COLUMNS = [
    "filename",
    "label",
    "source",
    "transactionDate",
    "plate",
    "province_code",
    "province_description",
    "brand_description",
    "colors_code",
    "colors_description",
    "vehicleClass",
]

RANDOM_SEED = 1337  # set None for non-deterministic runs
if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

# ----------------------------
# Project root detection
# ----------------------------
def find_project_root(start: Path) -> Path:
    """Find repo root so notebook works from synthetic_data/ or repo root."""
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "train" / "province_mapping.py").exists() and (p / "data").exists():
            return p
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / UPPER_SYNTH_REL
IMAGES_DIR = OUTPUT_DIR / UPPER_SYNTH_DATA_SUBDIR
LABELS_PATH = OUTPUT_DIR / "labels.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("IMAGES_DIR:", IMAGES_DIR)
print("LABELS_PATH:", LABELS_PATH)

# ----------------------------
# Fonts
# ----------------------------
def pick_font() -> ImageFont.ImageFont:
    preferred = "C:/Windows/Fonts/Sarun's ThangLuang.ttf"
    candidates = [
        preferred,
        "C:/Windows/Fonts/tahoma.ttf",
        "C:/Windows/Fonts/THSarabunNew.ttf",
        "C:/Windows/Fonts/THSarabunNew Bold.ttf",
    ]
    for path in candidates:
        if os.path.exists(path):
            try:
                return ImageFont.truetype(path, 38)  # slightly larger than lower line
            except OSError:
                continue
    print("Warning: Thai font not found, using default")
    return ImageFont.load_default()

font = pick_font()

# ----------------------------
# Palettes (reuse lower palette style)
# ----------------------------
palette = {
    "group1_personal": {
        "colors": [
            (255, 255, 255), (245, 245, 245), (230, 230, 230),
            (220, 225, 230), (210, 210, 210),
        ],
        "weight": 0.8,
    },
    "group2_taxi": {
        "colors": [
            (255, 235, 100), (255, 200, 90), (255, 180, 90),
            (150, 255, 150), (190, 240, 120),
        ],
        "weight": 0.1,
    },
    "group3_graphic": {
        "colors": [
            (255, 210, 230), (200, 220, 255), (255, 220, 170),
            (210, 200, 255), (180, 200, 210),
        ],
        "weight": 0.1,
    },
}

def choose_palette() -> Tuple[str, Tuple[int, int, int]]:
    groups = list(palette.keys())
    weights = [palette[g]["weight"] for g in groups]
    group = random.choices(groups, weights=weights, k=1)[0]
    color = random.choice(palette[group]["colors"])
    return group, color

# ----------------------------
# Character sets (uniform sampling per position)
# ----------------------------
DIGITS = list("0123456789")
THAI_CONSONANTS = [
    "ก", "ข", "ค", "ฆ", "ง", "จ", "ฉ", "ช", "ซ", "ฌ", "ญ",
    "ฎ", "ฏ", "ฐ", "ฑ", "ฒ", "ณ", "ด", "ต", "ถ", "ท",
    "ธ", "น", "บ", "ป", "ผ", "ฝ", "พ", "ฟ", "ภ", "ม",
    "ย", "ร", "ล", "ว", "ศ", "ษ", "ส", "ห", "ฬ", "อ", "ฮ",
]  # no ฅ ฃ

PREFIX_DIGIT_PROB = 0.2
LETTER_LEN_WEIGHTS = [(2, 0.8), (1, 0.1), (3, 0.1)]
SUFFIX_DIGIT_WEIGHTS = [(4, 0.75), (3, 0.2), (2, 0.05)]

def sample_plate_text() -> str:
    prefix = "" if random.random() > PREFIX_DIGIT_PROB else random.choice(DIGITS)
    letter_len = random.choices([l for l, _ in LETTER_LEN_WEIGHTS], weights=[w for _, w in LETTER_LEN_WEIGHTS], k=1)[0]
    letters = "".join(random.choice(THAI_CONSONANTS) for _ in range(letter_len))
    suffix_len = random.choices([l for l, _ in SUFFIX_DIGIT_WEIGHTS], weights=[w for _, w in SUFFIX_DIGIT_WEIGHTS], k=1)[0]
    digits = "".join(random.choice(DIGITS) for _ in range(suffix_len))
    return f"{prefix}{letters}{digits}"

# ----------------------------
# Drawing + augmentations
# ----------------------------
def draw_upper_line(text: str, bg_color: Tuple[int, int, int]) -> Image.Image:
    temp_img = Image.new("RGB", (1, 1))
    temp_draw = ImageDraw.Draw(temp_img)
    bbox = temp_draw.textbbox((0, 0), text, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]

    w = text_w + MARGIN_HORIZONTAL + random.randint(-5, 10)
    h = text_h + MARGIN_VERTICAL + random.randint(-3, 6)

    img = Image.new("RGB", (w, h), bg_color)
    draw = ImageDraw.Draw(img)
    draw.text((w // 2, h // 2), text, fill=(40, 40, 40), font=font, anchor="mm")
    return img

def apply_homography(img: np.ndarray) -> np.ndarray:
    height, width = img.shape[:2]
    src_points = np.float32([[0, 0], [width, 0], [width, height], [0, height]])
    distortion = random.uniform(0.03, 0.08)
    dst_points = np.float32([
        [random.uniform(0, width * distortion), random.uniform(0, height * distortion)],
        [width - random.uniform(0, width * distortion), random.uniform(0, height * distortion)],
        [width - random.uniform(0, width * distortion), height - random.uniform(0, height * distortion)],
        [random.uniform(0, width * distortion), height - random.uniform(0, height * distortion)],
    ])
    matrix = cv2.getPerspectiveTransform(src_points, dst_points)
    return cv2.warpPerspective(img, matrix, (width, height), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

def apply_augmentation(img: np.ndarray) -> np.ndarray:
    height, width = img.shape[:2]
    angle = random.uniform(-10, 10)
    rotation_matrix = cv2.getRotationMatrix2D((width / 2, height / 2), angle, 1.0)
    img = cv2.warpAffine(img, rotation_matrix, (width, height), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

    scale = random.uniform(0.85, 1.15)
    new_width = max(1, int(width * scale))
    new_height = max(1, int(height * scale))
    img = cv2.resize(img, (new_width, new_height), interpolation=cv2.INTER_CUBIC)

    if scale > 1.0:
        start_x = (new_width - width) // 2
        start_y = (new_height - height) // 2
        img = img[start_y : start_y + height, start_x : start_x + width]
    else:
        pad_x = (width - new_width) // 2
        pad_y = (height - new_height) // 2
        img = cv2.copyMakeBorder(
            img,
            pad_y,
            height - new_height - pad_y,
            pad_x,
            width - new_width - pad_x,
            cv2.BORDER_REPLICATE,
        )

    tx = random.randint(-18, 18)
    ty = random.randint(-8, 8)
    translation_matrix = np.float32([[1, 0, tx], [0, 1, ty]])
    img = cv2.warpAffine(img, translation_matrix, (width, height), borderMode=cv2.BORDER_REPLICATE)
    return img

def add_noise_and_blur(img: np.ndarray) -> np.ndarray:
    if random.random() < 0.3:
        if img.ndim == 3:
            noise = np.random.normal(0, 10, (img.shape[0], img.shape[1], 1)).astype(np.int16)
        else:
            noise = np.random.normal(0, 10, img.shape).astype(np.int16)
        img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    if random.random() < 1.0:
        kernel_size = random.choice([3, 5])
        img = cv2.GaussianBlur(img, (kernel_size, kernel_size), 0)

    if random.random() < 0.3:
        brightness = random.uniform(0.8, 1.2)
        img = cv2.convertScaleAbs(img, alpha=brightness, beta=0)
    return img

def apply_bicubic_interpolation(img: np.ndarray, scale: float = 0.5) -> np.ndarray:
    height, width = img.shape[:2]
    small = cv2.resize(img, (max(1, int(width * scale)), max(1, int(height * scale))), interpolation=cv2.INTER_CUBIC)
    return cv2.resize(small, (width, height), interpolation=cv2.INTER_CUBIC)

def thicken_text(image: np.ndarray) -> np.ndarray:
    k_size = 2
    kernel = np.ones((k_size, k_size), np.uint8)
    return cv2.erode(image, kernel, iterations=1)

def apply_heavy_motion_blur(image: np.ndarray) -> np.ndarray:
    kernel_size = random.choice([5, 7, 9, 11])
    kernel = np.zeros((kernel_size, kernel_size))
    kernel[int((kernel_size - 1) / 2), :] = np.ones(kernel_size)
    kernel /= kernel_size
    return cv2.filter2D(image, -1, kernel)

def fade_image(image: np.ndarray) -> np.ndarray:
    h, w, c = image.shape
    gray_val = random.randint(100, 180)
    gray_overlay = np.full((h, w, c), gray_val, dtype=np.uint8)
    alpha = random.uniform(0.1, 0.25)
    return cv2.addWeighted(image, 1 - alpha, gray_overlay, alpha, 0)

def apply_squash(img: np.ndarray) -> np.ndarray:
    h, w = img.shape[:2]
    squash_factor = random.uniform(0.6, 0.82)
    new_h = max(1, int(h * squash_factor))
    return cv2.resize(img, (w, new_h), interpolation=cv2.INTER_AREA)

# ----------------------------
# CSV helpers
# ----------------------------
def write_labels_header_if_missing():
    if not LABELS_PATH.exists():
        pd.DataFrame([], columns=LABEL_COLUMNS).to_csv(LABELS_PATH, index=False, encoding="utf-8-sig")

def append_rows(rows: List[Dict[str, object]]):
    if not rows:
        return
    df_new = pd.DataFrame(rows)
    for col in LABEL_COLUMNS:
        if col not in df_new.columns:
            df_new[col] = ""
    df_new = df_new[LABEL_COLUMNS]
    df_new.to_csv(LABELS_PATH, mode="a", header=False, index=False, encoding="utf-8-sig")

def existing_synth_count() -> int:
    return len(list(IMAGES_DIR.glob("synth_upper_*.jpg")))

# ----------------------------
# Main generation loop
# ----------------------------
def generate_upper_synthetic(n_samples: int = TARGET_SAMPLES, batch_flush: int = 2000):
    write_labels_header_if_missing()
    start_idx = existing_synth_count()
    current_idx = start_idx
    buffer: List[Dict[str, object]] = []
    generated = 0

    print(f"Starting from index: {current_idx}")
    print(f"Planned samples: {n_samples:,}")

    for _ in range(n_samples):
        while (IMAGES_DIR / f"synth_upper_{current_idx:06d}.jpg").exists():
            current_idx += 1

        plate_text = sample_plate_text()
        _, bg = choose_palette()
        img_pil = draw_upper_line(plate_text, bg)
        img_np = np.array(img_pil)

        img_np = apply_squash(img_np)
        if random.random() < 0.55:
            img_np = apply_homography(img_np)
        if random.random() < 0.65:
            img_np = apply_augmentation(img_np)
        if random.random() < 0.35:
            img_np = thicken_text(img_np)
        if random.random() < 0.8:
            img_np = apply_heavy_motion_blur(img_np)
        elif random.random() < 0.3:
            img_np = add_noise_and_blur(img_np)
        if random.random() < 0.65:
            img_np = fade_image(img_np)
        if random.random() < 0.3:
            img_np = add_noise_and_blur(img_np)
        if random.random() < 0.35:
            img_np = apply_bicubic_interpolation(img_np, scale=random.uniform(0.5, 0.85))

        img_np = cv2.resize(img_np, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

        fname = f"synth_upper_{current_idx:06d}.jpg"
        out_path = IMAGES_DIR / fname
        cv2.imwrite(str(out_path), cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR))

        buffer.append({
            "filename": fname,
            "label": plate_text,
            "source": "synthetic_upper",
            "transactionDate": "",
            "plate": plate_text,
            "province_code": "",
            "province_description": "",
            "brand_description": "",
            "colors_code": "",
            "colors_description": "",
            "vehicleClass": "1.0",
        })

        generated += 1
        current_idx += 1

        if len(buffer) >= batch_flush:
            append_rows(buffer)
            print(f"Flushed {len(buffer)} rows (total written: {generated:,})")
            buffer = []

    append_rows(buffer)
    print(f"Done. Generated total: {generated:,}")
    print(f"Images written to: {IMAGES_DIR}")
    print(f"Labels appended to: {LABELS_PATH}")

# Uncomment to run
generate_upper_synthetic(TARGET_SAMPLES)

PROJECT_ROOT: D:\CodingD\ALPR
OUTPUT_DIR: D:\CodingD\ALPR\data\plate_upper_synth_test
IMAGES_DIR: D:\CodingD\ALPR\data\plate_upper_synth_test\data
LABELS_PATH: D:\CodingD\ALPR\data\plate_upper_synth_test\labels.csv
Starting from index: 0
Planned samples: 5,000
Flushed 2000 rows (total written: 2,000)
Flushed 2000 rows (total written: 4,000)
Done. Generated total: 5,000
Images written to: D:\CodingD\ALPR\data\plate_upper_synth_test\data
Labels appended to: D:\CodingD\ALPR\data\plate_upper_synth_test\labels.csv
